# MoP + DivPO — Phase 3 Generation on Colab A100

This notebook runs the GPU-heavy generation stage for the baseline evaluation.

Use this while Kaggle is still training `single/all`:

- Preliminary 4-method run: `base`, `prompt_only`, `mop_sft`, `mop_divpo`
- Full 5-method run after Kaggle finishes: add `single_lora`

Runtime target: Colab A100.

---
## Cell 1 — Install Dependencies

Restart runtime once after this cell if Colab already imported Transformers/PEFT in this session.

In [ ]:
!pip install -q --upgrade transformers peft accelerate datasets huggingface_hub sentence-transformers trl
!pip uninstall -y -q torchao
print("Dependencies installed. Restart runtime if this was not a fresh runtime.")

---
## Cell 2 — Clone Branch and Configure Token

In [ ]:
import os
import sys
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/content/mop-divpo-llm-counter-argument"
REPO_URL = "https://github.com/DasonTio/mop-divpo-llm-counter-argument.git"
BRANCH = "feat/eval-pipeline"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch origin {BRANCH} --prune
!git -C {REPO_DIR} checkout -B {BRANCH} origin/{BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PYTHONPATH"] = f"{REPO_DIR}/src"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!git status --short --branch
!ls -la scripts/run_baseline_evaluation.py

---
## Cell 3 — Verify A100

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> GPU.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GB")
if "A100" not in torch.cuda.get_device_name(0):
    print("Warning: this is not an A100. It can still run, but estimates will be slower.")

---
## Cell 4 — Smoke Test Generation

Runs 2 prompts across the 4 methods that do not require `single/all`.

In [ ]:
!python scripts/run_baseline_evaluation.py \
  --only-generate \
  --limit-prompts 2 \
  --methods base prompt_only mop_sft mop_divpo \
  --outdir outputs/evaluation_prelim_a100_smoke

---
## Cell 5 — Preliminary 4-Method Generation

Use this while Kaggle is still training `single/all`. It writes `outputs/evaluation_prelim_a100/generations.jsonl`.

In [ ]:
!python scripts/run_baseline_evaluation.py \
  --only-generate \
  --methods base prompt_only mop_sft mop_divpo \
  --outdir outputs/evaluation_prelim_a100

---
## Cell 6 — Preliminary Automated Metrics

No API calls. Produces `baseline_table.{md,csv,json}` in the same output directory.

In [ ]:
!python scripts/run_baseline_evaluation.py \
  --skip-generation \
  --judge none \
  --methods base prompt_only mop_sft mop_divpo \
  --outdir outputs/evaluation_prelim_a100

!sed -n '1,120p' outputs/evaluation_prelim_a100/baseline_table.md

---
## Cell 7 — Check Whether `single/all` Exists

Run this after Kaggle finishes. Only run the full 5-method generation if this prints `single/all is available`.

In [ ]:
from huggingface_hub import list_repo_files
import os

files = list_repo_files(
    "DasonTio/mop-divpo-coauthor",
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
single_files = sorted(path for path in files if path.startswith("single/all/"))
print("\n".join(single_files) or "No single/all files found yet.")
if "single/all/adapter_config.json" not in single_files:
    raise RuntimeError("single/all is not available yet. Wait for Kaggle to finish and push.")
print("single/all is available.")

---
## Cell 8 — Full 5-Method Generation

Run only after Cell 7 succeeds. This is the generation file for the final paper table.

In [ ]:
!python scripts/run_baseline_evaluation.py \
  --only-generate \
  --methods base prompt_only single_lora mop_sft mop_divpo \
  --outdir outputs/evaluation_full_a100

---
## Cell 9 — Package Outputs for Download

In [ ]:
!zip -r evaluation_outputs_a100.zip outputs/evaluation_prelim_a100 outputs/evaluation_full_a100 2>/dev/null || true
print("Download evaluation_outputs_a100.zip from the Colab file browser if present.")